In [ ]:
# Data source switch
#   USE_RAW = True : read delay results from 2_prelimit_simulation/6_ECC_saved_results_full/ (full local results, NOT shipped)
#   USE_RAW = False: read delay results from 2_prelimit_simulation/4_ECC_saved_results/ (shipped)
#   (the delay_distribution_results CSVs are identical in both; plots are written next to them)
USE_RAW = False
RESULTS_BASE = "6_ECC_saved_results_full" if USE_RAW else "4_ECC_saved_results"

In [ ]:
# =============================================================================
# delay_distribution_analysis_cc3.ipynb
# Reads sim_one_combo CC3 outputs and plots delay-time histograms (4 routes).
# Handles abs/rel WMP variants → 5 policies × 2 + 8 non-WMP = 18 possible files.
# Missing files are skipped with a [skip] notice.
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ----- user-adjustable parameters ---------------------------------------------------------
RESULTS_DIR    = Path("../../2_prelimit_simulation/" + RESULTS_BASE + "/3_ECC10_opt_delay_distribution_5000p/delay_distribution_results/CC10")  # directory containing the CSV files
DELAY_SCALE    = 400.0       # delay is divided by this number when plotting
BUCKET_WIDTH   = 0.1         # must equal hist_bucket in one_combo_input_*.txt
PLOT_BIN_WIDTH = 0.005        # bin width for re-binning when plotting (delay-time unit)
DELAY_BOUND    = [0.15] * 10 + [0.6]   # per-route: route 1-3=0.15, route 4=0.225
X_LIM          = (0.0, 0.3)  # x axis range of the plot
N_ROUTES       = 11           # CC3 has 4 routes

# ----- save directory -------------------------------------------------------------
SAVE_DIR = Path("../../2_prelimit_simulation/" + RESULTS_BASE + "/3_ECC10_opt_delay_distribution_5000p/delay_distribution_plots/CC10")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ----- 18 (policy, variant) combinations -----------------------------------------
POLICY_VARIANTS = [
    ("proposed", "abs"), ("proposed", "rel"),
    ("A1", None), ("A2", None), ("A3", None), ("A4", None),
    ("B1", None), ("B2", None), ("B3", None), ("B4", None),
    ("C1", "abs"), ("C1", "rel"),
    ("C2", "abs"), ("C2", "rel"),
    ("D1", "abs"), ("D1", "rel"),
    ("D2", "abs"), ("D2", "rel"),
]

def variant_prefix(variant):
    return "" if variant is None else f"{variant}_"

def variant_label(name, variant):
    return name if variant is None else f"{name}-{variant}"

def safe_filename(name, variant):
    return name if variant is None else f"{name}_{variant}"

def find_csv(name, variant, suffix):
    p = RESULTS_DIR / f"{variant_prefix(variant)}{name}_{suffix}.csv"
    return p if p.exists() else None



In [ ]:
# =============================================================================
# 1. aggregate _summary.csv into one table
# =============================================================================
summaries = []
for name, variant in POLICY_VARIANTS:
    f = find_csv(name, variant, "summary")
    if f is None:
        print(f"[skip] no summary CSV for {variant_label(name, variant)}")
        continue
    df = pd.read_csv(f)
    df.insert(0, "filename", f.name)
    df.insert(1, "policy_label", variant_label(name, variant))
    summaries.append(df)

summary_table = pd.concat(summaries, ignore_index=True)
print(f"\nLoaded {len(summary_table)} policies into summary_table.")
display(summary_table[['policy_label', 'cost_mean']])

# optional: save the merged table
# summary_table.to_csv(SAVE_DIR / "all_policies_summary.csv", index=False)



In [ ]:
# =============================================================================
# 2. draw a 4-route delay-time histogram for each (policy, variant)
# =============================================================================

# ----- global font configuration ---------------------------------------------------------
plt.rcParams.update({
    'font.size':       25,
    'text.usetex':     True,
    'axes.titlesize':  25,
    'axes.labelsize':  25,
    'xtick.labelsize': 25,
    'ytick.labelsize': 25,
    'legend.fontsize': 20,
})

def plot_one_policy(name, variant):
    label = variant_label(name, variant)
    f = find_csv(name, variant, "delay_hist")
    if f is None:
        print(f"[skip] no histogram CSV for {label}")
        return

    df = pd.read_csv(f)
    if df.empty:
        print(f"[skip] empty histogram for {label}")
        return

    df = df.copy()
    df["delay"] = df["bucket_lower"] / DELAY_SCALE
    fine_bin    = BUCKET_WIDTH / DELAY_SCALE     # original bin width (delay-time unit)

    # --- remove the outer re-binning, do it on demand inside the route loop ---------------------
    # (the original df["new_bin"] = ... rebin = ... lines are removed)

    fig, axes = plt.subplots(N_ROUTES, 1, figsize=(6, 2.5*N_ROUTES), sharex=False)
    axes = axes.flatten()
    fig.suptitle(f"Policy: {label}", fontsize=18)

    for r in range(N_ROUTES):
        ax    = axes[r]
        col   = f"count_route{r}"
        bound = DELAY_BOUND[r]

        # each route's own plotting bin width
        bin_w = PLOT_BIN_WIDTH * 4 if r == N_ROUTES - 1 else PLOT_BIN_WIDTH

        # re-bin by bin_w (only for the current route column)
        tmp = df[["delay", col]].copy()
        tmp["new_bin"] = np.floor(tmp["delay"] / bin_w).astype(int)
        rebin = tmp.groupby("new_bin", as_index=False)[col].sum()
        rebin["bin_lower"] = rebin["new_bin"] * bin_w

        coarse_counts = rebin[col].values.astype(float)
        coarse_total  = coarse_counts.sum()
        if coarse_total == 0:
            ax.set_title(f"Route {r+1}  (no data)")
            ax.set_xlim(0.0, 1.0 if r == N_ROUTES - 1 else 0.3)
            continue
        density = coarse_counts / (coarse_total * bin_w)

        # fine-grained statistics (q99) stay unchanged, using df original bucket
        fine_counts = df[col].values.astype(float)
        fine_total  = fine_counts.sum()
        cum = np.cumsum(fine_counts) / fine_total
        q99_idx = min(int(np.searchsorted(cum, 0.99)), len(df) - 1)
        q99 = df["delay"].iloc[q99_idx]

        ax.bar(rebin["bin_lower"], density, width=bin_w, align="edge",
               color="lightgray", edgecolor="black", linewidth=0.5)
        ax.axvline(bound, color="black", linestyle=":",  linewidth=2, label="Bound")
        ax.axvline(q99,   color="red",   linestyle="--", linewidth=2, label="q99")

        if r == 0:
            ax.legend(loc="upper right")

        ax.set_title(f"Route {r+1}")
        ax.set_ylabel("Density")
        # ax.set_xlabel("Delay time")
        ax.set_xlim(0.0, 1.2 if r == N_ROUTES - 1 else 0.4)
        
        

    # # x axis labeled only on the bottom two subplots
    for ax in axes[N_ROUTES-1:]:
        ax.set_xlabel("Delay time")

    plt.tight_layout(rect=[0, 0, 1, 0.96])

    # ----- save ------------------------------------------------------------
    fname = safe_filename(name, variant)
    out_eps = SAVE_DIR / f"CC10_{fname}_delay_hist.eps"
    out_png = SAVE_DIR / f"CC10_{fname}_delay_hist.png"
    fig.savefig(out_eps, format="eps", bbox_inches="tight")
    fig.savefig(out_png, format="png", bbox_inches="tight")
    print(f"Saved: {out_eps.name}")

    plt.show()
    plt.close(fig)



In [ ]:
# draw all
for name, variant in POLICY_VARIANTS:
    plot_one_policy(name, variant)

In [ ]:
# =============================================================================
# merge 4 policies into one 11x4 grid figure
# =============================================================================

TARGET_POLICIES = [
    ("proposed", "abs", r"Proposed ($\tilde{\tau} = \mathrm{abs}$)"),
    ("proposed", "rel", r"Proposed ($\tilde{\tau} = \mathrm{rel}$)"),
    ("A1", None,        r"Benchmark A1"),
    ("A2", None,        r"Benchmark A2"),
]

# this grid plot uses a small font; use rc_context to temporarily override your global 25pt setting
GRID_RC = {
    'font.size':       12,
    'axes.titlesize':  12,
    'axes.labelsize':  12,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 12,
}


def _prepare_df(name, variant):
    """load and preprocess the histogram CSV of a single policy."""
    f = find_csv(name, variant, "delay_hist")
    if f is None:
        return None
    df = pd.read_csv(f)
    if df.empty:
        return None
    df = df.copy()
    df["delay"] = df["bucket_lower"] / DELAY_SCALE
    return df


def plot_combined_grid(policy_variants, save_name="combined_delay_hist"):
    # read all data first, exit immediately if any is missing
    dfs, labels = [], []
    for name, variant, lab in policy_variants:
        df = _prepare_df(name, variant)
        if df is None:
            print(f"[skip] no data for {lab}")
            return
        dfs.append(df)
        labels.append(lab)

    n_cols = len(policy_variants)

    with plt.rc_context(GRID_RC):
        fig, axes = plt.subplots(
            N_ROUTES, n_cols,
            figsize=(1.9 * n_cols, 0.8 * N_ROUTES),
            sharex='row',          # same route shares the x axis (route11 uses 0-1.2 separately)
            sharey=False,          # each subplot scales y on its own
            gridspec_kw={'hspace': 0.35, 'wspace': 0.22},
        )

        bound_h, q99_h = None, None  # keep a handle for the global legend

        for r in range(N_ROUTES):
            col      = f"count_route{r}"
            bound    = DELAY_BOUND[r]
            bin_w    = PLOT_BIN_WIDTH * 4 if r == N_ROUTES - 1 else PLOT_BIN_WIDTH
            xlim_max = 0.88 if r == N_ROUTES - 1 else 0.22

            for j, df in enumerate(dfs):
                ax = axes[r, j]

                # —— re-bin ——
                tmp = df[["delay", col]].copy()
                tmp["new_bin"] = np.floor(tmp["delay"] / bin_w).astype(int)
                rebin = tmp.groupby("new_bin", as_index=False)[col].sum()
                rebin["bin_lower"] = rebin["new_bin"] * bin_w
                coarse_counts = rebin[col].values.astype(float)
                coarse_total  = coarse_counts.sum()

                if coarse_total > 0:
                    density = coarse_counts / (coarse_total * bin_w)

                    fine_counts = df[col].values.astype(float)
                    fine_total  = fine_counts.sum()
                    cum     = np.cumsum(fine_counts) / fine_total
                    q99_idx = min(int(np.searchsorted(cum, 0.99)), len(df) - 1)
                    q99     = df["delay"].iloc[q99_idx]

                    ax.bar(rebin["bin_lower"], density, width=bin_w, align="edge",
                           color="lightgray", edgecolor="black", linewidth=0.3)
                    bh = ax.axvline(bound, color="black", linestyle=":",  linewidth=1.2)
                    qh = ax.axvline(q99,   color="red",   linestyle="--", linewidth=1.2)
                    if bound_h is None:
                        bound_h, q99_h = bh, qh

                ax.set_xlim(0.0, xlim_max)

                # force xticks to [0, 0.1, 0.2] for the first 10 routes
                if r < N_ROUTES - 1:
                    ax.set_xticks([0.0, 0.1, 0.2])
                else: 
                    ax.set_xticks([0.0, 0.4, 0.8])

                # not sharing y -> each subplot needs its own y tick numbers, shrink font size to avoid crowding
                ax.tick_params(labelsize=10)

            # R1-R9 hide x tick numbers (R10, R11 kept, because the two have different tick ranges)
            if r < N_ROUTES - 2:
                for j in range(n_cols):
                    axes[r, j].tick_params(labelbottom=False)

        # column titles: only on the top row
        for j, lab in enumerate(labels):
            axes[0, j].set_title(lab, pad=4)

        # row labels: leftmost column writes R1...R11 (horizontal, hugging the left)
        for r in range(N_ROUTES):
            axes[r, 0].set_ylabel(f"Route {r+1}", rotation=0,
                                  ha='right', va='center', labelpad=8)

        # x axis labels only on the bottom row
        for j in range(n_cols):
            axes[N_ROUTES - 1, j].set_xlabel("Delay time")

        # # global legend: centered just above the whole figure, close to it
        # if bound_h is not None:
        #     fig.legend(
        #         [bound_h, q99_h], ["Bound", "99-quantile"],
        #         loc='upper center', bbox_to_anchor=(0.5, 0.99),    # ← 0.975 → 0.99
        #         ncol=2, frameon=False,
        #     )

        # # add a "Density" on the left of the whole figure, close to it
        # fig.supylabel("Density", x=0.01, fontsize=9)            # ← 0.035 → 0.01
        # plt.tight_layout(rect=[0.09, 0, 1, 0.97])    
        # fig.supylabel("Density", x=0.01, fontsize=9)

        fig.subplots_adjust(
            left=0.10, right=0.99,
            top=0.955, bottom=0.04,
            hspace=0.35, wspace=0.22,
        )
        
        if bound_h is not None:
            fig.legend(
                [bound_h, q99_h], ["Delay bound", "99-quantile"],
                loc='lower center',                  # <- changed to lower center
                bbox_to_anchor=(0.8, 0.98),         # <- legend bottom edge at y=0.965, hugging axes top edge 0.955
                ncol=2, frameon=False,
            )

        # ----- save ------------------------------------------------------------
        out_eps = SAVE_DIR / f"CC10_{save_name}.eps"
        out_pdf = SAVE_DIR / f"CC10_{save_name}.pdf"
        out_png = SAVE_DIR / f"CC10_{save_name}.png"
        fig.savefig(out_eps, format="eps", bbox_inches="tight")
        fig.savefig(out_pdf, format="pdf", bbox_inches="tight")
        fig.savefig(out_png, format="png", bbox_inches="tight", dpi=300)
        print(f"Saved: {save_name}")
        plt.show()
        plt.close(fig)


plot_combined_grid(TARGET_POLICIES, save_name="combined_delay_hist")